# 2Dify AI — FULL-2D Video Reconstruction Worker

**Goal:** one source video upload → automatic 2D guide → generative video-to-video → strict full-frame 2D reconstruction → original audio.

The phone only uploads the video. A browser color filter is NOT used as the claimed AI conversion.


## 1. Install and upload ONE video


In [ ]:
!pip -q install -U gradio_client huggingface_hub imageio-ffmpeg pillow
from pathlib import Path
import os, subprocess
from PIL import Image, ImageEnhance, ImageFilter
from google.colab import files
from gradio_client import Client, handle_file
import imageio_ffmpeg
up=files.upload()
source_video=next(iter(up))
print('Source:',source_video)


## 2. Automatically create the 2D style guide
This guide is generated automatically from the first frame; the generative model does the actual visual reconstruction.


In [ ]:
ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()
raw='/content/2dify_first.png'
subprocess.run([ffmpeg,'-y','-i',source_video,'-frames:v','1','-vf','scale=768:-2',raw],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
img=Image.open(raw).convert('RGB')
img=img.filter(ImageFilter.MedianFilter(size=5))
img=ImageEnhance.Color(img).enhance(1.15)
img=ImageEnhance.Contrast(img).enhance(1.12)
img.save('/content/2d_style_guide.png',quality=95)
display(img)


## 3. Strict FULL-2D prompt


In [ ]:
STYLE_PROMPT='''FULL-FRAME 2D ANIMATED RECONSTRUCTION. Re-render EVERY visible region as coherent 2D illustrated artwork. Characters, faces, eyes, teeth, hair, skin, hands, clothes, accessories, animals, vehicles, furniture, tools, buildings, architecture, plants, sky, clouds, water, ground, shadows, highlights, reflections, particles and background MUST all be 2D illustrated shapes and linework. Use consistent hand-drawn/cartoon/anime/flat-illustration rendering across the entire frame. Preserve identity, pose, expression, gaze, body movement, interactions, lip-sync/performance and camera timing. Maintain temporal consistency: stable character design, outlines, colors and backgrounds. ZERO photorealism. ZERO live-action texture. ZERO 3D geometry. ZERO CGI. ZERO physically based materials. ZERO photographic skin/hair. ZERO realistic reflections. ZERO volumetric photographic lighting. Do not leave any original realistic region untouched. The COMPLETE FINAL FRAME must read as a single 2D animated illustration.'''
print(STYLE_PROMPT)


## 4. Connect to hosted ID-V2V and inspect its live API
The public Space can change its API or become unavailable, so this notebook inspects the current API instead of inventing an endpoint.


In [ ]:
SPACE='hugging-apps/id-v2v'
client=Client(SPACE)
api=client.view_api(all_endpoints=True,print_info=False)
print(api)


## 5. Run the real model
Use the exact endpoint and argument order printed by `view_api()`. Never fake a successful result when the Space is offline.


In [ ]:
# Example only — replace ENDPOINT and argument order using the live view_api() output.
# result=client.predict(handle_file(source_video),handle_file('/content/2d_style_guide.png'),STYLE_PROMPT,api_name=ENDPOINT)
# print(result)


## 6. Restore original audio after successful generation


In [ ]:
GENERATED=''
if GENERATED and os.path.exists(GENERATED):
    final='/content/2dify_full_2d.mp4'
    subprocess.run([ffmpeg,'-y','-i',GENERATED,'-i',source_video,'-map','0:v:0','-map','1:a?','-c:v','copy','-c:a','aac','-shortest',final],check=True)
    print('FINAL:',final)
else:
    print('No generated file yet. Run real inference first.')


## Quality gate
The target is full-frame 2D. If any obvious realistic/3D region survives, treat the conversion as unsuccessful rather than labeling it 100% 2D. Generative video cannot mathematically guarantee every pixel, so visual QA remains necessary.
